### 06.03. Setup functions and indexes

In [ ]:
# Set a connection with Azure OpenAI up
from llama_index.llms.azure_openai import AzureOpenAI
from llama_index.embeddings.azure_openai import AzureOpenAIEmbedding

from llama_index.core import Settings
import os
import nest_asyncio

nest_asyncio.apply()

# Enter your API key here
api_key = ""
endpoint = "https://agentic-ai-mvp.openai.azure.com/"
api_version = "2024-05-01-preview"

# "Function calling" available with GPT-4
Settings.llm=AzureOpenAI(
    model="gpt-4",
    deployment_name="agentai-gpt4",
    api_key=api_key,
    azure_endpoint=endpoint,
    api_version=api_version,
)

Settings.embed_model= AzureOpenAIEmbedding(
    model="text-embedding-ada-002",
    deployment_name="agentai-embedding",
    api_key=api_key,
    azure_endpoint=endpoint,
    api_version=api_version,
)


In [ ]:
from typing import List
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import  VectorStoreIndex
from llama_index.core.tools import QueryEngineTool

# -------------------------------------------------------------
# Tool 1 : Function to return the list of items for an eComm order
# -------------------------------------------------------------
def get_order_items(order_id: int) -> List[str] :
    """Given an order Id, this function returns the list of items 
    purchased with the specified order"""
    
    order_items = {
            1001: ["shirt","socks"],
            1002: ["pants","schawl"],
            1003: ["shirt","pants"]
        }
    if order_id in order_items.keys():
        return order_items[order_id]
    else:
        return []

# -------------------------------------------------------------
# Tool 2 : Function to return the delivery date for an eComm order
# -------------------------------------------------------------
def get_delivery_date(order_id: int) -> str:
    """Given an order Id, this function returns the delivery date 
    with the specified order"""

    delivery_dates = {
            1001: "10-Apr",
            1002: "12-Mar",
            1003: "08-Apr"       
    }
    if order_id in delivery_dates.keys():
        return delivery_dates[order_id]
    else:
        return []

# ----------------------------------------------------------------
# Tool 3 : Function to return the upper limit of "return days" for an item, per item name (to keep it less technical!)
# ----------------------------------------------------------------
def get_item_return_days(item: str) -> int :
    """Given an Item name, this function returns the number of days 
    allowed to return the specified product"""
    
    item_returns = {
            "shirt"     : 30,
            "socks"      : 15,
            "pants"   : 15,
            "schawl" : 5
    }
    if item in item_returns.keys():
        return item_returns[item]
    else:
        #Default
        return 45

# -------------------------------------------------------------
# Tool 4 : Vector DB, depicting customer support channels
# -------------------------------------------------------------
# Set a vector index up for return policy
support_docs=SimpleDirectoryReader(input_files=["customer-service.pdf"]).load_data()

splitter=SentenceSplitter(chunk_size=1024)
support_nodes=splitter.get_nodes_from_documents(support_docs)
support_index=VectorStoreIndex(support_nodes)
support_query_engine = support_index.as_query_engine()


### 06.04. Setup the Customer Service AI Agent

In [ ]:
from llama_index.core.tools import FunctionTool

# Create respective tools for 3 functions and 1 index
order_item_tool = FunctionTool.from_defaults(fn=get_order_items)
delivery_date_tool = FunctionTool.from_defaults(fn=get_delivery_date)
return_policy_tool = FunctionTool.from_defaults(fn=get_item_return_days)

support_tool = QueryEngineTool.from_defaults(
    query_engine=support_query_engine,
    description=(
        "Customer support, return policies and contact channels"
    ),
)

In [ ]:
from llama_index.core.agent import FunctionCallingAgentWorker
from llama_index.core.agent import AgentRunner
from llama_index.llms.openai import OpenAI

# Set a worker (Agent) in LlamaIndex with all the tools
# This is tool executor
agent_worker = FunctionCallingAgentWorker.from_tools(
    [order_item_tool, 
     delivery_date_tool,
     return_policy_tool,
     support_tool
    ], 
    llm=Settings.llm, 
    verbose=True
)
# Create an Agent Orchestrator, using LlamaIndex
agent = AgentRunner(agent_worker)


### 06.05. Using the customer service Agent

In [ ]:
# Retrieve the return policy details for an order, specified by order number
response = agent.query(
    "What is the return policy for the order number FO2003?"
)

print("\nOverall details of return policy:\n", response)

In [ ]:
# A three-part question
response = agent.query(
    "When are the delivery date and items for the order number FO2001 and how may I contact support?"
)

print("\nAggregated response:\n", response)

In [ ]:
# A question about an non-existent order, specified by order number
response = agent.query(
    "What is the return policy for order number FO2004?"
)

print("\nCollective answer:\n", response)